In [1]:
pip install pinecone python-dotenv

   ---------------------------------------- 0.0/742.8 kB ? eta -:--:--
   ---------------------------------------- 0.0/742.8 kB ? eta -:--:--
   -------------- ------------------------- 262.1/742.8 kB ? eta -:--:--
   ---------------------------- ----------- 524.3/742.8 kB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 742.8/742.8 kB 1.1 MB/s  0:00:00

   ----------------------------------------  0/11 [urllib3]
   ----------------------------------------  0/11 [urllib3]
   ----------------------------------------  0/11 [urllib3]
   ----------------------------------------  0/11 [urllib3]
   ----------------------------------------  0/11 [urllib3]
   ----------------------------------------  0/11 [urllib3]
   ----------------------------------------  0/11 [urllib3]
   --- ------------------------------------  1/11 [typing-extensions]
   ------- --------------------------------  2/11 [python-dotenv]
   ------- --------------------------------  2/11 [python-dotenv]
   --


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, uuid, itertools
from dotenv import load_dotenv
from docx import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone
 
# --- Config ---
load_dotenv()
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
INDEX_HOST  = "https://ecommerce-doc-9vbs4s0.svc.aped-4627-b74a.pinecone.io"
FILE_PATH   = r"data\\Generic E-Commerce Company_ Master Policy Compendium.docx"
NAMESPACE   = "default"
CHUNK_SIZE  = 500
CHUNK_OVERLAP = 100
BATCH_SIZE  = 96   # safe limit for integrated-embedding upsert_records
 
if not PINECONE_API_KEY:
    raise EnvironmentError("PINECONE_API_KEY not set. Add it to your .env file.")

In [3]:
from pinecone import Pinecone
import time

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "ecommerce-doc"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model": "llama-text-embed-v2",
            "field_map": {"text": "chunk_text"},
        },
    )
    print(f"Creating index '{index_name}'...")
    time.sleep(5)
else:
    print(f"Index '{index_name}' already exists.")

print("Index ready.")



Index 'ecommerce-doc' already exists.
Index ready.


In [4]:
def read_docx(path: str) -> str:
    doc = Document(path)
    return "\n".join(p.text for p in doc.paragraphs if p.text.strip())
 
 
def chunk_text(text: str) -> list[str]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", " ", ""],
    )
    return splitter.split_text(text)
 
 
def batched(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]
 
 
def main():
    # 1. Read
    print(f"Reading: {FILE_PATH}")
    full_text = read_docx(FILE_PATH)
    print(f"  {len(full_text):,} characters loaded")
 
    # 2. Chunk
    chunks = chunk_text(full_text)
    print(f"  {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")
 
    # 3. Build records — chunk_text field is mapped to the embed model's input
    records = [
        {
            "_id": str(uuid.uuid4()),
            "chunk_text": chunk,        # llama-text-embed-v2 field_map: {"text": "chunk_text"}
            "source": os.path.basename(FILE_PATH),
            "chunk_index": i,
        }
        for i, chunk in enumerate(chunks)
    ]
 
    # 4. Connect and upsert
    pc = Pinecone(api_key=PINECONE_API_KEY)
    index = pc.Index(host=INDEX_HOST)
 
    upserted = 0
    for batch in batched(records, BATCH_SIZE):
        index.upsert_records(NAMESPACE, batch)
        upserted += len(batch)
        print(f"  Upserted {upserted}/{len(records)} records...")
 
    print(f"\nDone! {upserted} chunks vectorized into '{INDEX_HOST}'")
 
 
if __name__ == "__main__":
    main()

Reading: data\\Generic E-Commerce Company_ Master Policy Compendium.docx
  47,994 characters loaded
  123 chunks (size=500, overlap=100)
  Upserted 96/123 records...
  Upserted 123/123 records...

Done! 123 chunks vectorized into 'https://ecommerce-doc-9vbs4s0.svc.aped-4627-b74a.pinecone.io'
